# Cross-fold validation

Splits train and val into 5 folds with near-duplicate images kept in the same fold, then trains once per fold. The folds are saved to `results/csvs/data/folds.csv`. Set `RUN_TRAINING = True` to train.

In [ ]:
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

import json
import warnings
import subprocess
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

import src.fine_tune.train as train_module
from src.fine_tune import Trainer, HybridModel
from src.fine_tune.losses import YOLOLoss
from src.data.preprocessing import ModelData, train_transform, eval_transform

from src.utils import parse_boxes, collate_fn
from src.config.paths import CSV_DIR, PLOT_DIR, DATASET_DIR, MODEL_DIR
from src.config.types import TrainingObjects

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

In [ ]:
N_FOLDS = 5
MAX_DISTANCE = 4 # max differing bits for two images to be duplicates
SEED = 42

# train_model logs CUDA memory but Mac doesnt support; ignore errors
if device.type != "cuda":
    torch.cuda.reset_peak_memory_stats = lambda: None
    torch.cuda.max_memory_allocated = lambda: 0

PARAMS = {
    "seed": SEED,
    "num_workers": 8 if device.type == "cuda" else 0,
    "pin_memory": device.type == "cuda",

    "batch_size": 8,
    "epochs": 8, # train.py uses 50, 8 for sake of time
    "accumulation_steps": 4,

    "learning_rate": 1e-3,
    "weight_decay": 1e-4,

    "model_name": "not saved",
}

# sample of each fold so all 5 fit in about an hour on a GPU, None uses the whole fold
TRAIN_IMAGES = 3000
VAL_IMAGES = 1000

RUN_TRAINING = False
RUN_NAME = "base" # names the folder, csv and plot of this run, e.g. weight_decay_1e-3 when PARAMS changes

In [ ]:
# train_model logs CUDA memory but Mac doesnt support; ignore errors
if device.type != "cuda":
    torch.cuda.reset_peak_memory_stats = lambda: None
    torch.cuda.max_memory_allocated = lambda: 0

warnings.filterwarnings("ignore", message="CUDA is not available")
warnings.filterwarnings("ignore", message="Encountered more than 100 detections")

# save_results writes to MODEL_DIR
RUN_DIR = MODEL_DIR / "cross_validation" / RUN_NAME
train_module.MODEL_DIR = RUN_DIR

CV_DIR = CSV_DIR / "cross_validation"
PLOT_DIR = PLOT_DIR / "cross_validation"
CV_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

print(device, RUN_DIR)

## Near-duplicate groups

In [ ]:
def image_hash(path):
    img = Image.open(path)
    img.draft("L", (64, 64)) # skips decoding the full resolution
    pixels = np.asarray(img.convert("L").resize((9, 8)), dtype=np.int16)
    return (pixels[:, 1:] > pixels[:, :-1]).flatten()


def hash_distances(a, b):
    # differing bits between every pair of images in a and b
    a, b = a.astype(np.float32), b.astype(np.float32)
    return a @ (1 - b).T + (1 - a) @ b.T


def near_duplicate_groups(hashes, max_distance):
    # union-find, the top of each chain is the group id
    parent = list(range(len(hashes)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]] # shortcut long chains
            i = parent[i]
        return i

    # 2000 rows at a time, 17k x 17k matrix needs over 1 GB otherwise
    for start in range(0, len(hashes), 2000):
        close = hash_distances(hashes[start:start + 2000], hashes) <= max_distance
        for i, j in zip(*np.nonzero(close)):
            parent[find(start + i)] = find(j)

    return np.array([find(i) for i in range(len(hashes))])


def assign_folds(groups, n_folds, seed):
    # biggest groups first into folds with least images to balance
    group_ids, sizes = np.unique(groups, return_counts=True)
    order = np.random.default_rng(seed).permutation(len(group_ids)) # random order for equal groups
    order = order[np.argsort(-sizes[order], kind="stable")]

    fold_sizes = np.zeros(n_folds, dtype=int)
    group_fold = {}
    for k in order:
        fold = fold_sizes.argmin()
        group_fold[group_ids[k]] = fold
        fold_sizes[fold] += sizes[k]

    return np.array([group_fold[group] for group in groups])

In [ ]:
df = pd.read_csv(CSV_DIR / "data/metadata.csv")
df["boxes"] = df["boxes"].apply(lambda value: parse_boxes(json.loads(value)))

# test is kept for the final model
pool = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
hashes = np.stack([image_hash(DATASET_DIR / path) for path in pool["path"]])

# too strict misses copies, too loose chains everything into one group
rows = []
for distance in [0, 2, 4, 6, 8]:
    sizes = np.unique(near_duplicate_groups(hashes, distance), return_counts=True)[1]
    rows.append({
        "max_distance": distance,
        "groups": len(sizes),
        "images_with_a_duplicate": sizes[sizes > 1].sum(),
        "largest_group": sizes.max(),
        "largest_group_share": sizes.max() / len(pool),
    })

pd.DataFrame(rows).round(3)

In [ ]:
pool["group"] = near_duplicate_groups(hashes, MAX_DISTANCE)
pool["fold"] = assign_folds(pool["group"].to_numpy(), N_FOLDS, SEED)

# every group must sit in exactly one fold
assert (pool.groupby("group")["fold"].nunique() == 1).all()

pool[["path", "split", "group", "fold"]].to_csv(CSV_DIR / "data/folds.csv", index=False)

# 5 images from each of the 3 biggest groups, one group per row
fig, axes = plt.subplots(3, 5, figsize=(15, 7))
for row, group in zip(axes, pool["group"].value_counts().index[:3]):
    for ax, path in zip(row, pool.loc[pool["group"] == group, "path"].sample(5, random_state=SEED)):
        ax.imshow(Image.open(DATASET_DIR / path))
        ax.set_title(Path(path).name, fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
pool["smoke_boxes"] = pool["boxes"].apply(lambda boxes: sum(box.category == 0 for box in boxes))
pool["fire_boxes"] = pool["boxes"].apply(lambda boxes: sum(box.category == 1 for box in boxes))
pool["source"] = pool["path"].str.extract(r"/([A-Za-z]+)\d+\.jpg$", expand=False)

# folds should have a similar size, class mix and sources
balance = pool.groupby("fold").agg(
    images=("path", "size"),
    groups=("group", "nunique"),
    no_boxes=("label", lambda label: (label == 0).mean()),
    smoke_boxes=("smoke_boxes", "sum"),
    fire_boxes=("fire_boxes", "sum"),
)
balance = balance.join(pd.crosstab(pool["fold"], pool["source"], normalize="index"))
balance.to_csv(CV_DIR / "fold_balance.csv")
balance.round(3)

In [ ]:
# how much the original splits leak
train_groups = set(pool.loc[pool["split"] == "train", "group"])
val = pool[pool["split"] == "val"]
print(f"{val['group'].isin(train_groups).sum()} of {len(val)} original val images share a group with a train image")

test = df[df["split"] == "test"]
test_hashes = np.stack([image_hash(DATASET_DIR / path) for path in test["path"]])
leaked = hash_distances(test_hashes, hashes).min(axis=1) <= MAX_DISTANCE
print(f"{leaked.sum()} of {len(test)} test images have a near-duplicate in train or val")

## Training

In [ ]:
git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()


def load_runs(folder):
    runs = {}
    for run_dir in sorted(folder.glob("*/history.csv")):
        with open(run_dir.parent / "metadata.json") as f:
            runs[run_dir.parent.name] = (json.load(f), pd.read_csv(run_dir))
    return runs


def to_records(split_df, n_images):
    if n_images is not None:
        split_df = split_df.sample(n_images, random_state=SEED)
    return list(split_df[["path", "boxes"]].itertuples(index=False, name=None))


def train_fold(fold):
    # same augmentation as scripts/training/train.py
    train_dataset = ModelData(
        DATASET_DIR, to_records(pool[pool["fold"] != fold], TRAIN_IMAGES), train_transform,
        flip_prob=0.5,
        zoom_prob=0.5, zoom_max=1.3
    )
    val_dataset = ModelData(DATASET_DIR, to_records(pool[pool["fold"] == fold], VAL_IMAGES), eval_transform)

    loader_args = dict(
        batch_size=PARAMS["batch_size"],
        collate_fn=collate_fn,
        num_workers=PARAMS["num_workers"],
        pin_memory=PARAMS["pin_memory"],
    )
    train_loader = DataLoader(train_dataset, shuffle=True, **loader_args)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_args)

    # same random head and shuffle order every fold, but different train/val split
    torch.manual_seed(SEED)
    model = HybridModel().to(device)

    optimiser = AdamW(
        [
            # pretrained backbone: smaller steps so the ImageNet features aren't wiped out
            {"params": model.backbone.parameters(), "lr": PARAMS["learning_rate"] * 0.1},
            # adapter, neck and head start from random weights, use full learning rate
            {"params": [p for name, p in model.named_parameters() if not name.startswith("backbone.")]},
        ],
        lr=PARAMS["learning_rate"],
        weight_decay=PARAMS["weight_decay"]
    )

    training_objs = TrainingObjects(
        model=model,
        optimizer=optimiser,
        criterion=YOLOLoss(model),
        scaler=torch.amp.GradScaler("cuda", enabled=device.type == "cuda"),
        device=device,
        accumulation_steps=PARAMS["accumulation_steps"],
    )

    return Trainer(objs=training_objs).train_model(
        epochs=PARAMS["epochs"],
        train_loader=train_loader,
        val_loader=val_loader
    )

In [ ]:
if RUN_TRAINING:
    done = [meta["notes"][0] for meta, _ in load_runs(RUN_DIR).values() if meta["notes"][-1] == "completed"]

    for fold in range(N_FOLDS):
        fold_note = f"fold {fold} of {N_FOLDS}"
        if fold_note in done:
            print(f"{fold_note} already saved, skipping")
            continue

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        results = train_fold(fold)

        model_info = {
            "timestamp": timestamp,
            "notes": [
                fold_note,
                f"train images {TRAIN_IMAGES or 'all'}, val images {VAL_IMAGES or 'all'}",
                "completed" if results.error is None else f"failed: {results.error!r}",
            ],
            "git_commit": git_commit,
            "parameters": PARAMS,
        }

        # only scores are needed, so weights aren't saved for every run
        results.best_state = None # no model.pt
        Trainer.save_results(results, model_info, f"fold_{fold}")

## Results

In [ ]:
runs = {name: run for name, run in load_runs(RUN_DIR).items() if run[0]["notes"][-1] == "completed"}

if runs:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    rows = []
    for name, (meta, history) in runs.items():
        fold = meta["notes"][0]
        line, = axes[0].plot(history["epoch"], history["train_loss"], label=f"{fold} train")
        axes[0].plot(history["epoch"], history["val_loss"], "--", color=line.get_color(), label=f"{fold} val")
        axes[1].plot(history["epoch"], history["map50_95"], marker="o", label=fold)

        # each fold is scored at its best epoch by mAP50-95, our main metric
        best = history.loc[history["map50_95"].idxmax()]
        rows.append({"fold": fold, "run": name, **best[["epoch", "map50", "map50_95", "recall50", "mar_100"]]})

    axes[0].set_title("Loss (dashed val)")
    axes[1].set_title("Validation mAP50-95 on the held-out fold")
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(linestyle="--", alpha=0.5)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{RUN_NAME}_folds.svg")
    plt.show()

    # best epoch per fold, then mean and std
    table = pd.DataFrame(rows).set_index("fold")
    metrics = table[["map50", "map50_95", "recall50", "mar_100"]]
    table = pd.concat([table, metrics.agg(["mean", "std"])])
    table.to_csv(CV_DIR / f"{RUN_NAME}_folds.csv")
    display(table.round(4))
else:
    print("No completed folds yet, set RUN_TRAINING = True to train")